# Raichur Cropping-Intensity Baseline — Map QA

Visual + statistical QA of `outputs/raichur_intensity_2024_25.tif` (Phase-1 peak-counting baseline, agri-year 2024-06-01 to 2025-05-31).

Classes: 0 fallow/non-crop, 1 single, 2 double, 3 triple+, 4 long-plateau (sugarcane/plantation flag), 255 nodata.

Checks: (1) class-area table + bar chart, (2) full-district class map, (3) command-area vs rainfed spot-check zoom, (4) sanity vs Raichur DES cropping-intensity magnitude.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

from cropint.config import load_config

cfg = load_config('../config/raichur.yaml')
MAP = Path('../outputs/raichur_intensity_2024_25.tif')
assert MAP.exists(), f'{MAP} not found — run scripts/generate_map.py first'

## 1. Class-area table

In [ ]:
csv = Path('../outputs/raichur_class_areas.csv')
if csv.exists():
    areas = pd.read_csv(csv)
else:
    with rasterio.open(MAP) as src:
        arr = src.read(1)
        px_km2 = abs(src.transform.a * src.transform.e) / 1e6
    vals, counts = np.unique(arr, return_counts=True)
    labels = {0:'fallow_or_noncrop',1:'single',2:'double',3:'triple_plus',4:'long_plateau_flag',255:'nodata'}
    areas = pd.DataFrame({'class':vals,'label':[labels.get(v,str(v)) for v in vals],
                          'pixel_count':counts,'area_km2':counts*px_km2})
    valid = areas[areas['class']!=255]['pixel_count'].sum()
    areas['pct_of_valid'] = np.where(areas['class']!=255, 100*areas['pixel_count']/valid, np.nan)
areas

In [ ]:
crop = areas[areas['class'].isin([1,2,3])]
plt.figure(figsize=(6,4))
plt.bar(crop['label'], crop['area_km2'], color=['#c2e699','#78c679','#238443'])
plt.ylabel('area (km$^2$)'); plt.title('Cropped area by intensity class — Raichur 2024-25')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 2. Full-district class map

In [ ]:
# discrete colormap: 0 tan, 1 light green, 2 mid green, 3 dark green, 4 magenta(plateau), 255 white
cmap = ListedColormap(['#d9c9a3','#c2e699','#78c679','#238443','#d01c8b','#ffffff'])
norm = BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5,255.5], cmap.N)
with rasterio.open(MAP) as src:
    arr = src.read(1)
fig, ax = plt.subplots(figsize=(11,10))
im = ax.imshow(arr, cmap=cmap, norm=norm)
cbar = fig.colorbar(im, ax=ax, ticks=[0,1,2,3,4,255], shrink=0.6)
cbar.ax.set_yticklabels(['0 fallow','1 single','2 double','3 triple+','4 plateau','255 nodata'])
ax.set_title('Raichur cropping intensity 2024-25 (10 m baseline)'); ax.axis('off'); plt.show()

## 3. Command-area vs rainfed spot-check
Zoom to a TLBC command-area window (Sindhanur, expect lots of class 2) and a rainfed window (Lingasugur, expect mostly class 1). Uses the geo-referenced transform to index by lon/lat.

In [ ]:
def window_around(src, lon, lat, half_km=6):
    row, col = src.index(lon, lat)
    d = int(half_km*1000 / abs(src.transform.a))
    return arr[max(0,row-d):row+d, max(0,col-d):col+d]

with rasterio.open(MAP) as src:
    cmd = window_around(src, 76.7560, 15.7702)   # Sindhanur command area
    rain = window_around(src, 76.5217, 16.1588)  # Lingasugur rainfed

fig, axes = plt.subplots(1,2, figsize=(13,6))
for ax, w, ttl in [(axes[0],cmd,'Sindhanur (command)'),(axes[1],rain,'Lingasugur (rainfed)')]:
    ax.imshow(w, cmap=cmap, norm=norm); ax.set_title(ttl); ax.axis('off')
plt.show()

for w, ttl in [(cmd,'command'),(rain,'rainfed')]:
    v,c = np.unique(w[w!=255], return_counts=True)
    print(ttl, {int(k):f'{100*n/c.sum():.0f}%' for k,n in zip(v,c)})

## 4. Sanity vs district statistics
Cropping intensity ≈ gross cropped area / net sown area = (1·A1 + 2·A2 + 3·A3) / (A1+A2+A3). Compare to the rough Raichur DES magnitude (~110–135% depending on source/year). This is an order-of-magnitude sanity check, not a formal accuracy assessment (that needs Digital Crop Survey ground truth).

In [ ]:
a = areas.set_index('class')['area_km2']
A1,A2,A3 = a.get(1,0), a.get(2,0), a.get(3,0)
nsa = A1+A2+A3
gca = 1*A1 + 2*A2 + 3*A3
print(f'Net sown (cropped) area (class 1-3): {nsa:,.0f} km2')
print(f'Gross cropped area: {gca:,.0f} km2')
print(f'Predicted cropping intensity: {100*gca/nsa:.1f}%  (DES rough magnitude ~110-135%)')
print(f'Long-plateau (sugarcane/plantation) flagged: {a.get(4,0):,.0f} km2')